# SentraGrade: Conditional GAN + ResNet50 Grader, with ML Testing
**Lab Evaluation 2: GAN model with another deep learning model, plus testing aspects on the case-study product.**

**Product context.** SentraGrade is an automated fruit and vegetable grading system that runs on a conveyor belt
(TR-6 dataset: Banana, Carrot, Guava, Indian Gooseberry, Mango, Tomato). The classes are heavily imbalanced
(Banana has 323 images, Tomato has 2244). A **conditional DCGAN** generates synthetic produce images per class. A
**ResNet50 classifier** (the grader) is the second deep learning model, and it consumes those images.

**Testing aspects covered (2 tests):**

| # | Test | Testing category | What it answers |
|---|------|------------------|-----------------|
| T1 | **FID (Fréchet Inception Distance)**, overall and per class, with real-vs-real floor and untrained-G ceiling | Correctness / output quality of a generative model (no ground truth, so we compare distributions) | Do the generated images look like real produce, and is quality uniform across classes? |
| T2 | **Back-to-back testing (TSTR / TRTS / Real+Synthetic)** on the same held-out real test set | Back-to-back testing from the ML-testing guide + downstream utility / synthetic-data fidelity | Is the synthetic data useful to the actual product model, and does adding it help or hurt? |

Each test has an explicit **acceptance criterion** and prints PASS/FAIL at the end.

## 0. Setup

In [ ]:
import os, glob, json, time, random, copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import torchvision.utils as vutils
from torchvision import models
from PIL import Image
from scipy import linalg
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt

# QUICK=True is a tiny smoke run that finishes in a few minutes. Use False for real results.
QUICK = False

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "NO GPU: turn on GPU T4 in Kaggle settings")

OUT = Path("/kaggle/working/gan_testing") if Path("/kaggle/working").exists() else Path("gan_testing_outputs")
OUT.mkdir(parents=True, exist_ok=True)

# ---- hyper-parameters ----
IMG = 64              # GAN resolution
NZ = 128              # latent size
GAN_EPOCHS = 2 if QUICK else 150
GAN_BS = 64
FID_EVERY = 1 if QUICK else 25
CLS_EPOCHS = 1 if QUICK else 8
CLS_SEEDS = [0] if QUICK else [0, 1, 2]   # repeated runs give mean ± std for the back-to-back comparison
CLS_RES = 128

## 1. Data
Upload `sentragrade_data.zip` (contains `resized/<Class>/*.jpg` and `manifest_full.csv`) as a Kaggle dataset.
The cell below finds it automatically under `/kaggle/input`.

**Leakage-safe split.** Photos of the same fruit on the same day are near-duplicates, so we split by
`group_id` (class + day). No day appears in both train and test.

In [ ]:
import zipfile

SEARCH_BASES = ["/kaggle/input", "/kaggle/working", ".", "..", str(Path("~/sentragrade_data").expanduser())]

def _search_manifest():
    for base in SEARCH_BASES:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, filenames in os.walk(base, followlinks=True):
            if "manifest_full.csv" in filenames and os.path.isdir(os.path.join(dirpath, "resized")):
                return Path(dirpath)
    return None

def find_data_root():
    root = _search_manifest()
    if root:
        return root
    # Kaggle sometimes keeps an uploaded .zip un-extracted, so unzip it into /kaggle/working ourselves
    for base in SEARCH_BASES[:1] + SEARCH_BASES[2:]:
        for z in glob.glob(f"{base}/**/*.zip", recursive=True):
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("manifest_full.csv") for n in zf.namelist()):
                    dest = Path("/kaggle/working/sentragrade_data") if Path("/kaggle/working").exists() else Path("sentragrade_data")
                    print(f"Extracting {z} -> {dest}")
                    zf.extractall(dest)
                    SEARCH_BASES.insert(0, str(dest))
                    root = _search_manifest()
                    if root:
                        return root
    print("Contents of /kaggle/input (first 40 entries):")
    for dirpath, dirnames, filenames in os.walk("/kaggle/input", followlinks=True):
        print(" ", dirpath, "| dirs:", dirnames[:6], "| files:", filenames[:4])
        if dirpath.count(os.sep) > 5:
            break
    raise FileNotFoundError("manifest_full.csv + resized/ not found. Attach the dataset made from sentragrade_data.zip "
                            "(right panel -> Add Input), then re-run this cell.")

DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)

man = pd.read_csv(DATA_ROOT / "manifest_full.csv")
man["path"] = [str(DATA_ROOT / "resized" / c / f) for c, f in zip(man.class_name, man.filename)]
man = man[man.path.map(os.path.exists)].drop_duplicates(["class_name", "filename"]).reset_index(drop=True)

CLASSES = sorted(man.class_name.unique())
NC = len(CLASSES)
cls2idx = {c: i for i, c in enumerate(CLASSES)}
man["y"] = man.class_name.map(cls2idx)

# per-class group split so every class appears in the test set
parts = []
for c, d in man.groupby("class_name"):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr, te = next(gss.split(d, groups=d.group_id))
    d = d.copy(); d["split"] = "train"; d.iloc[te, d.columns.get_loc("split")] = "test"
    parts.append(d)
man = pd.concat(parts).reset_index(drop=True)
if QUICK:
    man = man.groupby(["class_name", "split"]).head(40).reset_index(drop=True)

assert not set(man[man.split == "train"].group_id) & set(man[man.split == "test"].group_id), "group leakage!"
print(pd.crosstab(man.class_name, man.split, margins=True))

In [ ]:
def load_img(p, size=IMG):
    im = Image.open(p).convert("RGB")
    w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((size, size), Image.BICUBIC)
    return np.asarray(im)

def to_tensor(df):
    arr = np.stack([load_img(p) for p in df.path])                     # N,H,W,3 uint8
    x = torch.from_numpy(arr).permute(0, 3, 1, 2).float() / 127.5 - 1  # [-1, 1]
    return x, torch.tensor(df.y.values, dtype=torch.long)

t0 = time.time()
Xtr, ytr = to_tensor(man[man.split == "train"])
Xte, yte = to_tensor(man[man.split == "test"])
print(f"train {tuple(Xtr.shape)}  test {tuple(Xte.shape)}  ({time.time()-t0:.0f}s)")

def show_grid(x, nrow, title, fname=None):
    g = vutils.make_grid((x[:nrow * 8] + 1) / 2, nrow=nrow, padding=2).permute(1, 2, 0).numpy()
    plt.figure(figsize=(nrow * 1.3, 8 * 1.3 * len(x[:nrow*8]) / (nrow * 8) + 0.4)); plt.imshow(g); plt.axis("off"); plt.title(title)
    if fname: plt.savefig(OUT / fname, bbox_inches="tight", dpi=120)
    plt.show()

# real samples, 8 per class (one column per class)
idx = torch.cat([torch.where(ytr == c)[0][:8] for c in range(NC)])
show_grid(Xtr[idx].view(NC, 8, 3, IMG, IMG).transpose(0, 1).reshape(-1, 3, IMG, IMG), NC,
          "Real training images (columns: " + ", ".join(CLASSES) + ")", "real_samples.png")

## 2. Model 1: Conditional DCGAN
* **Generator:** `z (128) ⊕ class-embedding (50)`, then 5 transposed convolutions, giving a 64×64 RGB image (tanh).
* **Discriminator:** image ⊕ class-label plane, then 5 spectral-normalised convolutions, giving a real/fake logit.
* Stabilisers: spectral norm in D, one-sided label smoothing (0.9), horizontal flip on reals, and **class-balanced
  sampling** so minority classes (Banana, Mango) are not dropped. Minority modes are usually the first ones a GAN loses.

In [ ]:
class Generator(nn.Module):
    def __init__(self, nz=NZ, ncls=NC, ngf=64, emb=50):
        super().__init__()
        self.emb = nn.Embedding(ncls, emb)
        def up(i, o): return [nn.ConvTranspose2d(i, o, 4, 2, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True)]
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + emb, ngf * 8, 4, 1, 0, bias=False), nn.BatchNorm2d(ngf * 8), nn.ReLU(True),  # 4x4
            *up(ngf * 8, ngf * 4),   # 8
            *up(ngf * 4, ngf * 2),   # 16
            *up(ngf * 2, ngf),       # 32
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1), nn.Tanh())  # 64
    def forward(self, z, y):
        return self.net(torch.cat([z, self.emb(y)], 1)[:, :, None, None])

class Discriminator(nn.Module):
    def __init__(self, ncls=NC, ndf=64):
        super().__init__()
        SN = nn.utils.spectral_norm
        self.emb = nn.Embedding(ncls, IMG * IMG)
        def down(i, o): return [SN(nn.Conv2d(i, o, 4, 2, 1)), nn.LeakyReLU(0.2, True)]
        self.net = nn.Sequential(*down(4, ndf), *down(ndf, ndf * 2), *down(ndf * 2, ndf * 4), *down(ndf * 4, ndf * 8),
                                 SN(nn.Conv2d(ndf * 8, 1, 4, 1, 0)))
    def forward(self, x, y):
        return self.net(torch.cat([x, self.emb(y).view(-1, 1, IMG, IMG)], 1)).view(-1)

def weights_init(m):
    if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d)): nn.init.normal_(m.weight, 0, 0.02)
    elif isinstance(m, nn.BatchNorm2d): nn.init.normal_(m.weight, 1, 0.02); nn.init.zeros_(m.bias)

@torch.no_grad()
def generate(G, labels, bs=250):
    out = []
    for i in range(0, len(labels), bs):
        y = labels[i:i + bs].to(device)
        out.append(G(torch.randn(len(y), NZ, device=device), y).cpu())
    return torch.cat(out)

G = Generator().to(device); G.apply(weights_init)
D = Discriminator().to(device)
print(f"G params: {sum(p.numel() for p in G.parameters())/1e6:.2f}M | D params: {sum(p.numel() for p in D.parameters())/1e6:.2f}M")

## 3. FID utilities (used by Test T1 and for checkpoint selection)
FID = ‖μ_r − μ_g‖² + Tr(Σ_r + Σ_g − 2(Σ_r Σ_g)^{1/2}) on 2048-d Inception-v3 pool features. **Lower is better.**
The reference set is the real *training* images. Every FID we compare uses the same sample size N,
because FID is biased by N.

In [ ]:
try:
    inception = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
except Exception as e:
    raise RuntimeError("Could not download Inception-v3 weights. In Kaggle turn ON 'Internet' in the notebook settings.") from e
inception.fc = nn.Identity()
inception = inception.eval().to(device)
_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def inception_feats(x, bs=100):
    feats = []
    for i in range(0, len(x), bs):
        b = (x[i:i + bs].to(device) + 1) / 2
        b = F.interpolate(b, size=(299, 299), mode="bilinear", align_corners=False)
        feats.append(inception((b - _MEAN) / _STD).double().cpu())
    return torch.cat(feats).numpy()

def fid(f1, f2):
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    covmean = linalg.sqrtm(s1 @ s2)
    if not np.isfinite(covmean).all():
        off = np.eye(s1.shape[0]) * 1e-6
        covmean = linalg.sqrtm((s1 + off) @ (s2 + off))
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean.real))

t0 = time.time()
feat_train = inception_feats(Xtr)
feat_test = inception_feats(Xte)
print(f"Inception features: train {feat_train.shape}, test {feat_test.shape} ({time.time()-t0:.0f}s)")

## 4. Train the GAN (tracks FID every few epochs and keeps the best checkpoint)

In [ ]:
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

counts = torch.bincount(ytr, minlength=NC).float()
sampler = WeightedRandomSampler((1.0 / counts)[ytr], num_samples=len(ytr), replacement=True)
gan_dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=GAN_BS, sampler=sampler, drop_last=True)

fixed_y = torch.arange(NC).repeat(8)
fixed_z = torch.randn(len(fixed_y), NZ, device=device)
hist = {"d_loss": [], "g_loss": [], "D(x)": [], "D(G(z))": []}
fid_hist, best_fid, best_G = [], float("inf"), None

t0 = time.time()
for ep in range(1, GAN_EPOCHS + 1):
    s = {k: 0.0 for k in hist}; n = 0
    for xb, yb in gan_dl:
        xb, yb = xb.to(device), yb.to(device)
        flip = torch.rand(len(xb), device=device) < 0.5
        xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
        z = torch.randn(len(xb), NZ, device=device)
        yf = torch.randint(0, NC, (len(xb),), device=device)
        fake = G(z, yf)

        d_real, d_fake = D(xb, yb), D(fake.detach(), yf)
        loss_D = bce(d_real, torch.full_like(d_real, 0.9)) + bce(d_fake, torch.zeros_like(d_fake))
        opt_D.zero_grad(set_to_none=True); loss_D.backward(); opt_D.step()

        g_out = D(fake, yf)
        loss_G = bce(g_out, torch.ones_like(g_out))
        opt_G.zero_grad(set_to_none=True); loss_G.backward(); opt_G.step()

        s["d_loss"] += loss_D.item(); s["g_loss"] += loss_G.item()
        s["D(x)"] += torch.sigmoid(d_real).mean().item(); s["D(G(z))"] += torch.sigmoid(d_fake).mean().item(); n += 1
    for k in hist: hist[k].append(s[k] / n)

    if ep % FID_EVERY == 0 or ep == GAN_EPOCHS:
        f = fid(inception_feats(generate(G, yte)), feat_train)   # same N and class mix as the real test set
        fid_hist.append((ep, f))
        if f < best_fid:
            best_fid, best_G = f, copy.deepcopy(G.state_dict())
        print(f"epoch {ep:3d} | D {hist['d_loss'][-1]:.3f} G {hist['g_loss'][-1]:.3f} | "
              f"D(x) {hist['D(x)'][-1]:.2f} D(G(z)) {hist['D(G(z))'][-1]:.2f} | FID {f:.1f} | {time.time()-t0:.0f}s")

G.load_state_dict(best_G)
torch.save(G.state_dict(), OUT / "generator_best.pt")
print(f"Best FID during training: {best_fid:.1f} (checkpoint restored)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(hist["d_loss"], label="D loss"); ax[0].plot(hist["g_loss"], label="G loss"); ax[0].set_title("GAN losses"); ax[0].legend()
ax[1].plot(hist["D(x)"], label="D(x) real"); ax[1].plot(hist["D(G(z))"], label="D(G(z)) fake"); ax[1].set_title("Discriminator outputs"); ax[1].legend()
ax[2].plot(*zip(*fid_hist), "o-"); ax[2].set_title("FID vs epoch (lower = better)"); ax[2].set_xlabel("epoch")
for a in ax[:2]: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(OUT / "gan_training_curves.png", dpi=120); plt.show()

with torch.no_grad():
    fixed = G(fixed_z, fixed_y.to(device)).cpu()
show_grid(fixed, NC, "Generated images (columns: " + ", ".join(CLASSES) + ")", "generated_samples.png")

---
# TEST T1: Output correctness via FID
Generative models have no ground truth, so we test the **distribution**:

* **Floor (real vs real):** FID(real test, real train). This is the best FID achievable given dataset variation.
* **Model:** FID(generated, real train), using the same N and the same class mix as the test set.
* **Ceiling / oracle sanity check:** FID(untrained G, real train). If the metric can't separate this from the model, the test itself is broken.
* **Per-class FID:** shows whether quality is uniform across produce types (a bias check, since minority classes are often worse).

**Acceptance criteria:**
* T1a: FID_model < 0.5 × FID_untrained (the GAN learned a lot more than noise).
* T1b: worst-class FID ≤ 2 × best-class FID (no class is badly under-served).

In [ ]:
set_seed(SEED)
G_untrained = Generator().to(device); G_untrained.apply(weights_init)

fid_floor = fid(feat_test, feat_train)
fid_model = fid(inception_feats(generate(G, yte)), feat_train)
fid_untrained = fid(inception_feats(generate(G_untrained, yte)), feat_train)

ytr_np, yte_np = ytr.numpy(), yte.numpy()
rows = []
for c, name in enumerate(CLASSES):
    ref = feat_train[ytr_np == c]
    n_c = int((yte_np == c).sum())
    f_real = fid(feat_test[yte_np == c], ref)
    f_gen = fid(inception_feats(generate(G, torch.full((n_c,), c))), ref)
    rows.append({"class": name, "n_train": int((ytr_np == c).sum()), "N": n_c,
                 "FID_real_floor": round(f_real, 1), "FID_generated": round(f_gen, 1), "gap": round(f_gen - f_real, 1)})
per_class_fid = pd.DataFrame(rows)

t1_overall = pd.DataFrame({"setting": ["Real test vs real train (floor)", "Trained cGAN vs real train", "Untrained G vs real train (ceiling)"],
                           "FID": [round(fid_floor, 1), round(fid_model, 1), round(fid_untrained, 1)]})
print("== T1 overall FID =="); display(t1_overall)
print("== T1 per-class FID =="); display(per_class_fid)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(t1_overall.setting.str.split(" vs").str[0], t1_overall.FID, color=["#4c9a2a", "#2a6f9a", "#9a2a2a"])
ax[0].set_title("Overall FID (lower = better)")
w = 0.38; xs = np.arange(NC)
ax[1].bar(xs - w/2, per_class_fid.FID_real_floor, w, label="real floor")
ax[1].bar(xs + w/2, per_class_fid.FID_generated, w, label="generated")
ax[1].set_xticks(xs); ax[1].set_xticklabels(CLASSES, rotation=20); ax[1].set_title("Per-class FID"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "t1_fid.png", dpi=120); plt.show()

T1a = fid_model < 0.5 * fid_untrained
T1b = per_class_fid.FID_generated.max() <= 2 * per_class_fid.FID_generated.min()

---
# Model 2: ResNet50 grader (the product's classifier)
ImageNet-pretrained ResNet50, the same backbone as the original SentraGrade grader, with conv1/layer1/layer2 **frozen** and layer3/layer4 plus a new 6-way head fine-tuned (the same freeze as the original). Images are upsampled to 128 px, and the loss uses balanced class weights.
Every variant uses identical hyper-parameters and seeds, so **the training data is the only thing that changes**.

In [ ]:
IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IM_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def prep(x, train):
    x = (x.to(device) + 1) / 2
    if train:
        flip = torch.rand(len(x), device=device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(3), x)
    x = F.interpolate(x, size=(CLS_RES, CLS_RES), mode="bilinear", align_corners=False)
    return (x - IM_MEAN) / IM_STD

def make_classifier():
    try:
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    except Exception:
        print("WARNING: no internet, ResNet50 trains from scratch"); m = models.resnet50(weights=None)
    # same freeze as the original SentraGrade ResNet50 (freeze_until="layer2"):
    # stem + layer1 + layer2 frozen, layer3 + layer4 + head fine-tuned
    for name in ["conv1", "bn1", "layer1", "layer2"]:
        for p in getattr(m, name).parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, NC)
    return m.to(device)

def train_classifier(X, y, seed):
    set_seed(seed)
    m = make_classifier()
    cnt = torch.bincount(y, minlength=NC).float().clamp(min=1)
    crit = nn.CrossEntropyLoss(weight=(cnt.sum() / (NC * cnt)).to(device))
    dl = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CLS_EPOCHS * len(dl))
    m.train()
    for ep in range(CLS_EPOCHS):
        for xb, yb in dl:
            loss = crit(m(prep(xb, True)), yb.to(device))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    return m.eval()

@torch.no_grad()
def predict(m, X, bs=256):
    return torch.cat([m(prep(X[i:i + bs], False)).argmax(1).cpu() for i in range(0, len(X), bs)]).numpy()

def scores(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro"),
            **{f"recall_{c}": r for c, r in zip(CLASSES, recall_score(y_true, y_pred, average=None, labels=range(NC)))}}

---
# TEST T2: Back-to-back testing / downstream utility
Same model, same seeds, same **real held-out test set**. Only the training data differs:

| Variant | Train on | Test on | Meaning |
|---|---|---|---|
| **TRTR** | Real | Real | Baseline grader |
| **TSTR** | Synthetic only (same size and class mix as real) | Real | Does the GAN capture the features that matter for grading? |
| **TRTS** | Real | Synthetic | Are generated images recognisable as their class? (conditional correctness) |
| **R+S** | Real + class-balanced synthetic | Real | Product use case: GAN augmentation for minority classes |

**Acceptance criteria:**
* T2a: TSTR accuracy ≥ 0.70 × TRTR accuracy (synthetic-data utility ratio)
* T2b: TRTS accuracy ≥ 0.70 (generated images carry the correct class)
* T2c: macro-F1(R+S) ≥ macro-F1(TRTR) − 0.01 (augmentation must not regress the product)

In [ ]:
set_seed(SEED)
Xsyn, ysyn = generate(G, ytr), ytr.clone()                     # TSTR set: mirrors real train
per_cls = len(ytr) // NC
y_bal = torch.arange(NC).repeat_interleave(per_cls)
X_aug, y_aug = torch.cat([Xtr, generate(G, y_bal)]), torch.cat([ytr, y_bal])
y_trts = torch.arange(NC).repeat_interleave(max(len(yte) // NC, 1))   # TRTS set: balanced synthetic test set
X_trts = generate(G, y_trts)

results, preds_store = [], {}
t0 = time.time()
for seed in CLS_SEEDS:
    m_real = train_classifier(Xtr, ytr, seed)
    p = predict(m_real, Xte);   results.append({"variant": "TRTR", "seed": seed, **scores(yte_np, p)}); preds_store[("TRTR", seed)] = p
    p = predict(m_real, X_trts); results.append({"variant": "TRTS", "seed": seed, **scores(y_trts.numpy(), p)})
    del m_real

    m_syn = train_classifier(Xsyn, ysyn, seed)
    p = predict(m_syn, Xte);    results.append({"variant": "TSTR", "seed": seed, **scores(yte_np, p)}); preds_store[("TSTR", seed)] = p
    del m_syn

    m_aug = train_classifier(X_aug, y_aug, seed)
    p = predict(m_aug, Xte);    results.append({"variant": "R+S", "seed": seed, **scores(yte_np, p)}); preds_store[("R+S", seed)] = p
    del m_aug
    torch.cuda.empty_cache()
    print(f"seed {seed} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(results)
order = ["TRTR", "TSTR", "TRTS", "R+S"]
summary = res.groupby("variant").agg(["mean", "std"]).drop(columns="seed").reindex(order)
display(summary[["acc", "macro_f1"]].round(4))
print("Per-class recall (mean over seeds): checks whether minority classes benefit")
display(res.groupby("variant")[[f"recall_{c}" for c in CLASSES]].mean().reindex(order).round(3))

In [ ]:
mean = res.groupby("variant")[["acc", "macro_f1"]].mean().reindex(order)
std = res.groupby("variant")[["acc", "macro_f1"]].std().reindex(order).fillna(0)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
xs = np.arange(len(order)); w = 0.38
ax[0].bar(xs - w/2, mean.acc, w, yerr=std.acc, capsize=4, label="accuracy")
ax[0].bar(xs + w/2, mean.macro_f1, w, yerr=std.macro_f1, capsize=4, label="macro-F1")
ax[0].set_xticks(xs); ax[0].set_xticklabels(order); ax[0].set_ylim(0, 1); ax[0].legend(); ax[0].set_title("T2 back-to-back comparison")

rec = res.groupby("variant")[[f"recall_{c}" for c in CLASSES]].mean()
for v in ["TRTR", "R+S"]:
    ax[1].plot(CLASSES, rec.loc[v].values, "o-", label=v)
ax[1].set_title("Per-class recall: real vs real+synthetic"); ax[1].legend(); ax[1].tick_params(axis="x", rotation=20)

cm = confusion_matrix(yte_np, preds_store[("TSTR", CLS_SEEDS[0])], labels=range(NC), normalize="true")
ax[2].imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax[2].set_xticks(range(NC)); ax[2].set_xticklabels(CLASSES, rotation=45, ha="right"); ax[2].set_yticks(range(NC)); ax[2].set_yticklabels(CLASSES)
for i in range(NC):
    for j in range(NC): ax[2].text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax[2].set_title("TSTR confusion matrix (rows = true)")
plt.tight_layout(); plt.savefig(OUT / "t2_back_to_back.png", dpi=120); plt.show()

acc = mean.acc; f1m = mean.macro_f1
T2a = acc["TSTR"] >= 0.70 * acc["TRTR"]
T2b = acc["TRTS"] >= 0.70
T2c = f1m["R+S"] >= f1m["TRTR"] - 0.01

---
# Test report

In [ ]:
report = pd.DataFrame([
    ["T1a", "FID: model vs untrained G", f"{fid_model:.1f} < 0.5 × {fid_untrained:.1f}", T1a],
    ["T1b", "FID: class uniformity", f"max {per_class_fid.FID_generated.max():.1f} ≤ 2 × min {per_class_fid.FID_generated.min():.1f}", T1b],
    ["T2a", "TSTR utility ratio", f"{acc['TSTR']:.3f} / {acc['TRTR']:.3f} = {acc['TSTR']/acc['TRTR']:.2f} ≥ 0.70", T2a],
    ["T2b", "TRTS class correctness", f"{acc['TRTS']:.3f} ≥ 0.70", T2b],
    ["T2c", "Augmentation non-regression", f"F1 {f1m['R+S']:.3f} ≥ {f1m['TRTR']:.3f} − 0.01", T2c],
], columns=["id", "test", "measured", "pass"])
report["result"] = report["pass"].map({True: "PASS", False: "FAIL"})
display(report.drop(columns="pass"))

json.dump({
    "fid": {"floor_real_vs_real": fid_floor, "model": fid_model, "untrained": fid_untrained, "best_during_training": best_fid,
            "per_class": per_class_fid.to_dict("records"), "history": fid_hist},
    "back_to_back": res.to_dict("records"),
    "report": report.drop(columns="pass").to_dict("records"),
    "config": {"GAN_EPOCHS": GAN_EPOCHS, "CLS_EPOCHS": CLS_EPOCHS, "CLS_SEEDS": CLS_SEEDS, "IMG": IMG, "QUICK": QUICK},
}, open(OUT / "test_results.json", "w"), indent=2, default=float)
print("Saved figures + test_results.json + generator_best.pt to", OUT)

### How to read the results
* **T1 (FID):** the gap between *generated* and the *real floor* is how far the GAN is from real data. Per-class gaps show
  which produce types the GAN serves poorly. These are usually the minority classes, which is the GAN-amplifies-imbalance bias risk.
* **T2 (back-to-back):** TSTR well below TRTR means the synthetic images look plausible but miss grading-relevant
  detail. If **R+S** raises recall on Banana and Mango without lowering macro-F1, GAN augmentation is worth shipping
  in SentraGrade. If it regresses (T2c FAIL), the test has stopped a harmful change before deployment, which is the point of the test.
* Caveat: FID here uses torchvision's Inception-v3 and 64×64 images, so absolute values are not comparable with published
  FID numbers. Only the relative comparisons inside this notebook are meaningful.